In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

DATA_DIR = "/kaggle/input/datasets/dhoogla/cicids2017"

PARQUET_FILES = [
    "Benign-Monday-no-metadata.parquet",
    "Botnet-Friday-no-metadata.parquet",
    "Bruteforce-Tuesday-no-metadata.parquet",
    "DDoS-Friday-no-metadata.parquet",
    "DoS-Wednesday-no-metadata.parquet",
    "Infiltration-Thursday-no-metadata.parquet",
    "Portscan-Friday-no-metadata.parquet",
    "WebAttacks-Thursday-no-metadata.parquet",
]

In [ ]:
def infer_attack_category_from_filename(fname: str) -> str:
    base = os.path.basename(fname).lower()
    if "benign" in base:
        return "Normal"
    if "botnet" in base:
        return "Bot"
    if "bruteforce" in base:
        return "BruteForce"
    if "ddos" in base:
        return "DDoS"
    if "dos" in base:
        return "DoS"
    if "portscan" in base:
        return "PortScan"
    if "webattacks" in base or "webattacks" in base:
        return "WebAttack"
    if "infiltration" in base:
        return "Infiltration"
    return "Unknown"

dfs = []
for fname in PARQUET_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    df_part = pd.read_parquet(fpath)
    df_part["attack_category"] = infer_attack_category_from_filename(fname)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
df.head()

In [ ]:
print("Shape (rows, columns):", df.shape)

memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Approx. memory usage: {memory_mb:.2f} MB")

print("\nDataFrame info():")
df.info()

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe().T.head()

In [ ]:
EDA_SAMPLE_SIZE = 500_000
if len(df) > EDA_SAMPLE_SIZE:
    df_eda = df.sample(EDA_SAMPLE_SIZE, random_state=42)
else:
    df_eda = df.copy()

df_eda.shape

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.countplot(
    data=df,
    x="attack_category",
    order=df["attack_category"].value_counts().index,
    palette="tab10"
)

plt.title("Class distribution (attack categories)")
plt.xlabel("Attack category")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")

for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{height:,}",
        (p.get_x() + p.get_width() / 2., height),
        ha="center", va="bottom", fontsize=10, rotation=90
    )

plt.tight_layout()
plt.savefig("class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

df["attack_category"].value_counts(normalize=True).mul(100).round(2)